# Data centre locations: OpenStreetMap and PeeringDB

Facility locations are collected from two open sources:

- **OpenStreetMap (OSM)**: objects tagged `telecom=data_center` or `building=data_center`, downloaded through the Overpass API. Buildings drawn as outlines also give the footprint.
- **PeeringDB**: colocation and interconnection facilities registered by their operators.

Both lists are cleaned, merged into one table of facilities and shown on an interactive map. The output, `data/clean/datacenters_locations.csv`, is the input of the second notebook.

Downloads are kept in `data/raw/`, so later runs read the saved files instead of downloading again. Paths are set relative to the project folder (the one containing `.git`), so the notebook saves to the right place no matter which folder it is run from.

In [1]:
# !pip install requests pandas numpy ftfy unidecode reverse_geocoder pycountry folium ipywidgets

import html
import json
import os
import re
import time
from pathlib import Path

import folium
import ftfy
import ipywidgets as widgets
import numpy as np
import pandas as pd
import pycountry
import requests
import reverse_geocoder as rg
from folium.plugins import FastMarkerCluster
from IPython.display import display
from unidecode import unidecode

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

In [2]:
# Project folder: the first folder upwards that contains .git, so paths work from anywhere
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
RAW = ROOT / "data" / "raw"       # downloads, never edited by hand
CLEAN = ROOT / "data" / "clean"   # tables produced by the notebooks

PEERINGDB_FILE = RAW / "peeringdb" / "peeringdb_facilities.json"
OSM_TILE_DIR = RAW / "osm_tiles"
OUTPUT_FILE = CLEAN / "datacenters_locations.csv"

REFRESH_PEERINGDB = False   # True downloads PeeringDB again instead of reading the saved file
MATCH_RADIUS_M = 250        # a PeeringDB and an OSM site closer than this are treated as one facility

PEERINGDB_URL = "https://www.peeringdb.com/api/fac"
OVERPASS_URLS = [           # public Overpass servers, the next one is tried when a server is busy
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]
HEADERS = {"User-Agent": "datacenter-inventory-research/1.0"}

# A single worldwide Overpass query times out, so the world is split into 30 boxes (south, west, north, east)
LAT_BANDS = [(-60, -15), (-15, 20), (20, 40), (40, 55), (55, 85)]
LON_BANDS = [(-180, -100), (-100, -30), (-30, 20), (20, 60), (60, 110), (110, 180)]
WORLD_BOXES = [(s, w, n, e) for s, n in LAT_BANDS for w, e in LON_BANDS]

for folder in (PEERINGDB_FILE.parent, OSM_TILE_DIR, CLEAN):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project folder: {ROOT}")

Project folder: /Users/shirindehghannezhad/Documents/Data-Center


### Functions

In [3]:
def fix_text(value):
    # Repairs broken encoding ("GonzÃ¡lez" -> "González"), collapses whitespace
    # and turns placeholders such as "n/a" into a missing value
    if not isinstance(value, str):
        return value
    value = " ".join(ftfy.fix_text(value).split())
    return None if value.lower() in {"", "none", "nan", "null", "-", "n/a"} else value


def fix_text_columns(table):
    text_columns = table.select_dtypes(include=["object", "string"]).columns
    table[text_columns] = table[text_columns].apply(lambda column: column.map(fix_text))
    return table


def to_latin(value):
    # Latin-script copy of names written in other scripts, e.g. "Москва" -> "Moskva"
    return unidecode(value) if isinstance(value, str) else value


def distance_m(lat1, lon1, lat2, lon2):
    # Haversine distance in metres; works element-wise on arrays
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6_371_000 * 2 * np.arcsin(np.sqrt(a))

In [4]:
def overpass_query(box):
    s, w, n, e = box
    return f"""
    [out:json][timeout:300];
    (
      nwr["telecom"="data_center"]({s},{w},{n},{e});
      nwr["building"="data_center"]({s},{w},{n},{e});
    );
    out tags geom;
    """


def download_overpass(query, retries=6):
    # Busy servers answer 429 or 504; each retry waits a little longer and moves to the next server
    for attempt in range(retries):
        url = OVERPASS_URLS[attempt % len(OVERPASS_URLS)]
        try:
            response = requests.post(url, data={"data": query}, headers=HEADERS, timeout=360)
            if response.status_code == 200:
                response.encoding = "utf-8"
                return response.json().get("elements", [])
            print(f"    {url} returned {response.status_code}")
        except (requests.RequestException, ValueError) as error:
            print(f"    {url} failed: {error}")
        time.sleep(20 * (attempt + 1))
    return None


def ring_area_m2(ring):
    # Shoelace formula on a local flat projection; accurate enough at building scale
    lon = np.array([p[0] for p in ring])
    lat = np.array([p[1] for p in ring])
    x = lon * 111_320 * np.cos(np.radians(lat.mean()))
    y = lat * 110_540
    return abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))) / 2


def parse_osm_element(element):
    tags = element.get("tags", {})
    ring = None

    if element["type"] == "node":
        lat, lon = element["lat"], element["lon"]
    else:
        if element["type"] == "way":
            ring = [(p["lon"], p["lat"]) for p in element.get("geometry", [])]
        else:
            # Relations: the largest outer ring stands for the building
            outer_rings = [[(p["lon"], p["lat"]) for p in member["geometry"]]
                           for member in element.get("members", [])
                           if member.get("role") == "outer" and member.get("geometry")]
            ring = max(outer_rings, key=ring_area_m2) if outer_rings else None
        if not ring or len(ring) < 4:
            return None
        lon, lat = np.mean(ring[:-1], axis=0)   # the last point repeats the first

    def first_tag(*keys):
        return next((tags[k] for k in keys if tags.get(k)), None)

    street = " ".join(filter(None, [tags.get("addr:street"), tags.get("addr:housenumber")]))
    return {
        "source": "osm",
        "osm_id": f"{element['type']}/{element['id']}",
        "lat": round(float(lat), 6),
        "lon": round(float(lon), 6),
        "name": first_tag("name:en", "int_name", "name"),
        "operator": first_tag("operator:en", "operator", "owner", "brand"),
        "website": first_tag("website", "contact:website"),
        "address": street or None,
        "city": first_tag("addr:city:en", "addr:city"),
        "postcode": tags.get("addr:postcode"),
        "start_date": tags.get("start_date"),
        "building_levels": tags.get("building:levels"),
        "footprint_m2": round(ring_area_m2(ring)) if ring else None,
        "footprint": json.dumps([[round(x, 6), round(y, 6)] for x, y in ring]) if ring else None,
    }

In [5]:
def match_sites(peeringdb, osm, radius_m):
    # Nearest-neighbour matching, one to one: candidate pairs within the radius are
    # taken from the closest upwards, and each site is used at most once
    pairs = []
    for i, (lat, lon) in enumerate(zip(peeringdb["lat"], peeringdb["lon"])):
        distances = distance_m(lat, lon, osm["lat"].values, osm["lon"].values)
        for j in np.flatnonzero(distances <= radius_m):
            pairs.append((distances[j], i, j))

    used_pdb, used_osm, matches = set(), set(), []
    for dist, i, j in sorted(pairs):
        if i not in used_pdb and j not in used_osm:
            used_pdb.add(i)
            used_osm.add(j)
            matches.append((i, j, round(dist)))
    return pd.DataFrame(matches, columns=["pdb_row", "osm_row", "match_distance_m"])

### Get the data

#### PeeringDB

The full facility list comes from the public API; no account is needed. The raw answer is saved as JSON.

In [6]:
if REFRESH_PEERINGDB or not PEERINGDB_FILE.exists():
    response = requests.get(PEERINGDB_URL, params={"status": "ok"}, headers=HEADERS, timeout=120)
    response.raise_for_status()
    response.encoding = "utf-8"
    with open(PEERINGDB_FILE, "w", encoding="utf-8") as f:
        json.dump(response.json()["data"], f, ensure_ascii=False)

with open(PEERINGDB_FILE, encoding="utf-8") as f:
    facilities = json.load(f)

peeringdb = pd.DataFrame({
    "source": "peeringdb",
    "peeringdb_id": [f"fac/{fac['id']}" for fac in facilities],
    "lat": [fac.get("latitude") for fac in facilities],
    "lon": [fac.get("longitude") for fac in facilities],
    "name": [fac.get("name") for fac in facilities],
    "operator": [fac.get("org_name") for fac in facilities],
    "website": [fac.get("website") for fac in facilities],
    "address": [" ".join(filter(None, [fac.get("address1"), fac.get("address2")])) for fac in facilities],
    "city": [fac.get("city") for fac in facilities],
    "postcode": [fac.get("zipcode") for fac in facilities],
    "country_reported": [fac.get("country") for fac in facilities],
    "net_count": [fac.get("net_count") for fac in facilities],
    "ix_count": [fac.get("ix_count") for fac in facilities],
})
peeringdb = fix_text_columns(peeringdb)

### OpenStreetMap

Each of the 30 boxes is saved to `data/raw/osm_tiles/` as soon as it arrives. If the download stops, running the cell again continues with the missing boxes.

In [7]:
for i, box in enumerate(WORLD_BOXES, 1):
    tile_file = OSM_TILE_DIR / f"tile_{i:02d}.json"
    if tile_file.exists():
        continue
    print(f"box {i}/{len(WORLD_BOXES)} {box}")
    elements = download_overpass(overpass_query(box))
    if elements is None:
        print("    not downloaded, run the cell again later")
        continue
    with open(tile_file, "w", encoding="utf-8") as f:
        json.dump(elements, f, ensure_ascii=False)
    time.sleep(5)

osm_rows, seen = [], set()
for i in range(1, len(WORLD_BOXES) + 1):
    tile_file = OSM_TILE_DIR / f"tile_{i:02d}.json"
    if not tile_file.exists():
        continue
    with open(tile_file, encoding="utf-8") as f:
        for element in json.load(f):
            key = (element["type"], element["id"])
            if key in seen:   # objects on a box edge are returned by both boxes
                continue
            seen.add(key)
            row = parse_osm_element(element)
            if row:
                osm_rows.append(row)

osm = fix_text_columns(pd.DataFrame(osm_rows))

### Raw data

In [8]:
print(f"PeeringDB: {len(peeringdb)} facilities, OpenStreetMap: {len(osm)} objects")
display(peeringdb.head(3), osm.drop(columns="footprint").head(3))

missing = pd.concat({"PeeringDB": peeringdb.isna().mean(), "OSM": osm.isna().mean()}, axis=1)
(missing * 100).round(1).rename_axis("column").add_suffix(" % missing")

PeeringDB: 5885 facilities, OpenStreetMap: 4713 objects


,source,peeringdb_id,lat,lon,name,operator,website,address,city,postcode,country_reported,net_count,ix_count
0,peeringdb,fac/1,39.016363,-77.459023,"Equinix DC1-DC15,DC21-DC22 - Ashburn","Equinix, Inc.",http://www.equinix.com/,21715 Filigree Ct Building F,Ashburn,20147-6205,US,525,9
1,peeringdb,fac/2,37.723407,-122.397923,Digital Realty SFO (200 Paul),Digital Realty,https://www.digitalrealty.com,200 Paul Ave,San Francisco,94110,US,44,2
2,peeringdb,fac/4,32.800955,-96.819550,Equinix DA1 - Dallas,"Equinix, Inc.",https://www.equinix.com/locations/americas-col...,1950 N Stemmons Fwy,Dallas,75207-3137,US,167,5


,source,osm_id,lat,lon,name,operator,website,address,city,postcode,start_date,building_levels,footprint_m2
0,osm,node/5321056354,-32.996325,-71.507912,Servicio Televisión por Cable,None,None,Pasaje Patrullero Campos pasaje 1,Viña del Mar,None,None,None,NaN
1,osm,node/6519075547,-26.184087,-59.363239,Nodo FO REFSA Tel - VILLA 213,REFSA,None,None,None,None,None,None,NaN
2,osm,node/7034694131,-26.175320,-58.833466,Nodo FO REFSA Tel - IRIGOYEN,REFSA,None,None,None,None,None,None,NaN


,PeeringDB % missing,OSM % missing
column,,
source,0.0,0.0
peeringdb_id,0.0,NaN
lat,10.3,0.0
lon,10.3,0.0
name,0.0,23.9
operator,0.0,40.4
website,1.1,77.3
address,1.0,59.2
city,0.0,66.0


### Merge

A PeeringDB facility and an OSM object within `MATCH_RADIUS_M` of each other are treated as the same site. In a merged site the PeeringDB name, operator and address are kept, since operators maintain them, while OSM adds the footprint and the building centre as coordinates. Unmatched sites from both sources are kept as they are.

In [9]:
# Sites without coordinates cannot be matched or mapped
peeringdb = peeringdb.dropna(subset=["lat", "lon"]).reset_index(drop=True)
peeringdb[["lat", "lon"]] = peeringdb[["lat", "lon"]].astype(float).round(6)

matches = match_sites(peeringdb, osm, MATCH_RADIUS_M)
pdb_matched = peeringdb.loc[matches["pdb_row"]].reset_index(drop=True)
osm_matched = osm.loc[matches["osm_row"]].reset_index(drop=True)

merged = pdb_matched.combine_first(osm_matched.drop(columns=["lat", "lon", "source"]))
merged[["lat", "lon"]] = np.where(osm_matched[["footprint"]].notna().values,
                                  osm_matched[["lat", "lon"]].values, pdb_matched[["lat", "lon"]].values)
merged["source"] = "peeringdb+osm"
merged["match_distance_m"] = matches["match_distance_m"].values

datacenters = pd.concat([
    merged,
    peeringdb.drop(index=matches["pdb_row"]),
    osm.drop(index=matches["osm_row"]),
], ignore_index=True)

### Clean

- coordinates outside the valid range or at 0,0 (a common placeholder) are dropped, as are exact duplicates
- country, region and nearest town are taken from the coordinates, so every site has them even where the source left them empty
- Latin-script copies are added for names, operators and cities
- the OSM `start_date` is reduced to a year
- each site gets a stable ID

In [10]:
valid = datacenters["lat"].between(-90, 90) & datacenters["lon"].between(-180, 180)
not_null_island = ~((datacenters["lat"].abs() < 0.01) & (datacenters["lon"].abs() < 0.01))
datacenters = datacenters[valid & not_null_island].drop_duplicates(subset=["lat", "lon", "name"]).reset_index(drop=True)

# Offline reverse geocoding (GeoNames): country code, first-level region and nearest town
places = rg.search(list(zip(datacenters["lat"], datacenters["lon"])), mode=1, verbose=False)
datacenters["country_code"] = [p["cc"] for p in places]
datacenters["region"] = [fix_text(p["admin1"]) for p in places]
datacenters["nearest_town"] = [fix_text(p["name"]) for p in places]

country_names = {c.alpha_2: getattr(c, "common_name", c.name) for c in pycountry.countries}
country_names["XK"] = "Kosovo"   # not an ISO country, but used by GeoNames
datacenters["country"] = datacenters["country_code"].map(country_names)

for column in ["name", "operator", "city"]:
    datacenters[f"{column}_latin"] = datacenters[column].map(to_latin)
datacenters["city_latin"] = datacenters["city_latin"].fillna(datacenters["nearest_town"])

# start_date comes in many forms ("2015", "2015-06", "~2015", "before 2010"); the first plausible year is kept
current_year = pd.Timestamp.now().year
years = datacenters["start_date"].str.extract(r"((?:19|20)\d{2})", expand=False).astype(float)
datacenters["osm_start_year"] = years.where(years.between(1960, current_year)).astype("Int64")

for column in ["net_count", "ix_count", "building_levels", "footprint_m2"]:
    datacenters[column] = pd.to_numeric(datacenters[column], errors="coerce")

datacenters = datacenters.sort_values(["country", "city_latin", "name_latin"]).reset_index(drop=True)
datacenters.insert(0, "dc_id", [f"DC{i:06d}" for i in range(1, len(datacenters) + 1)])

column_order = [
    "dc_id", "name", "name_latin", "operator", "operator_latin", "source", "peeringdb_id", "osm_id",
    "lat", "lon", "country", "country_code", "region", "city", "city_latin", "nearest_town", "address",
    "postcode", "website", "net_count", "ix_count", "osm_start_year", "building_levels",
    "footprint_m2", "footprint", "match_distance_m",
]
datacenters = datacenters[column_order]
datacenters.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

In [11]:
print(f"{len(datacenters)} sites in {datacenters['country'].nunique()} countries, saved to {OUTPUT_FILE.relative_to(ROOT)}")
display(datacenters["source"].value_counts().to_frame("sites"))
datacenters.drop(columns="footprint").head()

8739 sites in 160 countries, saved to data/clean/datacenters_locations.csv


,sites
source,
peeringdb,4026
osm,3460
peeringdb+osm,1253


,dc_id,name,name_latin,operator,operator_latin,source,peeringdb_id,osm_id,lat,lon,country,country_code,region,city,city_latin,nearest_town,address,postcode,website,net_count,ix_count,osm_start_year,building_levels,footprint_m2,match_distance_m
0,DC000001,ISPPro.al,ISPPro.al,None,None,osm,NaN,way/950023771,42.121369,19.561237,Albania,AL,Shkoder,None,Gruemire,Gruemire,None,None,None,NaN,NaN,<NA>,NaN,109.0,NaN
1,DC000002,Tirana DataCom,Tirana DataCom,Albanian Telecommunications Union SH. P.K.,Albanian Telecommunications Union SH. P.K.,peeringdb,fac/5286,NaN,41.370100,19.698340,Albania,AL,Tirane,Kashar,Kashar,Berxull,Rruga Industriale,None,http://www.atu.al/,2.0,0.0,<NA>,NaN,NaN,NaN
2,DC000003,DC-T01 iHost.AL,DC-T01 iHost.AL,iHost.AL,iHost.AL,peeringdb,fac/15711,NaN,41.337336,19.838920,Albania,AL,Tirane,Tirana,Tirana,Tirana,"Rruga Kongresi i Manastirit, Njesia Bashkiake ...",1001,https://ihost.al,2.0,0.0,<NA>,NaN,NaN,NaN
3,DC000004,Datacenter Albania,Datacenter Albania,Host.AL Shpk,Host.AL Shpk,peeringdb,fac/15686,NaN,41.329279,19.827882,Albania,AL,Tirane,Tirana,Tirana,Tirana,"Tefta Tashko, 23",1002,https://host.al,2.0,0.0,<NA>,NaN,NaN,NaN
4,DC000005,One Albania,One Albania,One Albania,One Albania,peeringdb,fac/2213,NaN,41.367987,19.703438,Albania,AL,Tirane,Tirana,Tirana,Kashar,"Autostrada Tirane-Durres, Km 7, Kashar",1001,http://www.one.al,0.0,0.0,<NA>,NaN,NaN,NaN


### Map by country

The map opens on all sites worldwide, grouped into clusters that split up when zooming in. Choosing a country redraws it with that country's sites and, where OSM has them, the building outlines. Colours show the source, and clicking a site opens its details.

In [12]:
import html
import json

from branca.element import MacroElement
from folium.plugins import MarkerCluster
from jinja2 import Template

SOURCE_COLOURS = {"peeringdb": "#2a6fdb", "osm": "#e8702a", "peeringdb+osm": "#1f9e6e"}
SOURCE_LABELS = {"peeringdb": "PeeringDB", "osm": "OpenStreetMap", "peeringdb+osm": "Both"}


def popup_text(site):
    rows = [
        ("Operator", site.operator_latin),
        ("Location", ", ".join(str(v) for v in [site.city_latin, site.country] if pd.notna(v))),
        ("Address", site.address),
        ("Source", SOURCE_LABELS[site.source]),
        ("Footprint", f"{site.footprint_m2:,.0f} m²" if pd.notna(site.footprint_m2) else None),
        ("Networks / IXPs", f"{site.net_count:.0f} / {site.ix_count:.0f}" if pd.notna(site.net_count) else None),
        ("ID", site.dc_id),
    ]
    table = "".join(f"<tr><td style='color:#666;padding-right:8px'>{label}</td><td>{html.escape(str(value))}</td></tr>"
                    for label, value in rows if pd.notna(value) and value != "")
    name = site.name_latin if pd.notna(site.name_latin) else "Unnamed site"
    return f"<b>{html.escape(name)}</b><table style='font-size:12px;margin-top:4px'>{table}</table>"


# One compact row per site; the map's own script draws the markers from these rows
map_rows = [{
    "lat": site.lat, "lon": site.lon, "country": site.country,
    "colour": SOURCE_COLOURS[site.source],
    "label": site.name_latin if pd.notna(site.name_latin) else "Unnamed site",
    "popup": popup_text(site),
    "outline": [[lat, lon] for lon, lat in json.loads(site.footprint)] if pd.notna(site.footprint) else None,
} for site in datacenters.itertuples()]

country_counts = datacenters["country"].value_counts().sort_index()
country_options = [["Worldwide", f"Worldwide ({len(datacenters)})"]] + \
                  [[country, f"{country} ({n})"] for country, n in country_counts.items()]


class CountryFilter(MacroElement):
    # Country dropdown inside the map. "Worldwide" shows all sites in clusters;
    # a country shows its sites one by one, with their OSM outlines, and zooms to them.
    _template = Template("""
    {% macro script(this, kwargs) %}
    (function () {
        var map = {{ this._parent.get_name() }};
        var rows = {{ this.rows }};
        var options = {{ this.options }};
        var worldLayer = {{ this.cluster }};
        var countryLayer = L.featureGroup();

        function marker(row) {
            return L.circleMarker([row.lat, row.lon], {radius: 5, color: "white", weight: 1,
                                  fillColor: row.colour, fillOpacity: 0.9})
                    .bindPopup(row.popup, {maxWidth: 320}).bindTooltip(row.label);
        }
        rows.forEach(function (row) { worldLayer.addLayer(marker(row)); });

        function show(country) {
            map.removeLayer(countryLayer);
            map.removeLayer(worldLayer);
            if (country === "Worldwide") {
                map.addLayer(worldLayer);
                map.setView([25, 10], 2);
                return;
            }
            countryLayer.clearLayers();
            rows.filter(function (row) { return row.country === country; }).forEach(function (row) {
                if (row.outline) {
                    L.polygon(row.outline, {color: row.colour, weight: 1, fillOpacity: 0.3}).addTo(countryLayer);
                }
                marker(row).addTo(countryLayer);
            });
            map.addLayer(countryLayer);
            map.fitBounds(countryLayer.getBounds(), {maxZoom: 12, padding: [40, 40]});
        }

        var picker = L.control({position: "topright"});
        picker.onAdd = function () {
            var box = L.DomUtil.create("div", "leaflet-bar");
            box.style.cssText = "background:white;padding:4px 6px;font:13px sans-serif";
            var select = L.DomUtil.create("select", "", box);
            options.forEach(function (o) { select.add(new Option(o[1], o[0])); });
            select.onchange = function () { show(this.value); };
            L.DomEvent.disableClickPropagation(box);
            L.DomEvent.disableScrollPropagation(box);
            return box;
        };
        picker.addTo(map);
        show("Worldwide");
    })();
    {% endmacro %}
    """)

    def __init__(self, rows, options, cluster):
        super().__init__()
        # "</" is escaped so popup HTML cannot close the surrounding <script> tag
        self.rows = json.dumps(rows, ensure_ascii=False).replace("</", "<\\/")
        self.options = json.dumps(options, ensure_ascii=False)
        self.cluster = cluster.get_name()


site_map = folium.Map(location=[25, 10], zoom_start=2, control_scale=True)
world_clusters = MarkerCluster(control=False)   # filled by the script; created here so folium loads the cluster library
world_clusters.add_to(site_map)
CountryFilter(map_rows, country_options, world_clusters).add_to(site_map)

legend = "".join(f"<div><span style='display:inline-block;width:10px;height:10px;border-radius:50%;"
                 f"background:{c};margin-right:6px'></span>{SOURCE_LABELS[s]}</div>"
                 for s, c in SOURCE_COLOURS.items())
site_map.get_root().html.add_child(folium.Element(
    f"<div style='position:fixed;bottom:20px;left:10px;z-index:1000;background:white;padding:6px 10px;"
    f"border-radius:4px;font:12px sans-serif;box-shadow:0 1px 4px rgba(0,0,0,.3)'>{legend}</div>"))

map_figure = folium.Figure(height=600)   # fixed height; without it some editors show an empty frame
map_figure.add_child(site_map)
map_figure